# TileDB-SOMA storage companion

Throwaway notebook for getting a closer feel for the structures described in `../../wiki/concepts/tiledb-soma-storage.md`.

The goal is not analysis. The goal is to inspect the layers:

1. Census collection
2. SOMA experiment
3. `obs` and `var` dataframes
4. RNA `X` sparse arrays
5. TileDB schema / fragment directories on S3

Most cells are metadata-only. The only expression-matrix read is guarded by `RUN_LIVE_X_QUERY = False` because even tiny-looking gene filters can cause large cell-major fragment reads.

## 0. Imports and knobs

Use this from the repo environment. If imports fail, install through `uv add`, not `pip install`.

In [ ]:
import importlib.metadata as md
import json
import textwrap
import time
from pathlib import Path

import cellxgene_census
import pandas as pd
import tiledb
import tiledbsoma

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

ORGANISM = "homo_sapiens"
MEASUREMENT = "RNA"
X_LAYER = "raw"
GENES = ["ADORA1", "ADORA2A", "ADORA2B", "ADORA3"]

# Keep these false while skimming. Flip them only when you intentionally want live reads.
RUN_OBS_ENUMERATION = False
RUN_LIVE_X_QUERY = False


## 1. Open the Census handle

`open_soma()` opens a lazy handle. This should not download the expression matrix. The context mirrors the fetch script's generous S3 timeouts so structure inspection is less likely to die on a slow network.

In [ ]:
census = cellxgene_census.open_soma()
census

## 2. Census top level

This is the first wiki layer: a remote collection with metadata in `census_info` and per-organism experiments in `census_data`.

In [ ]:
def describe_soma_node(name, obj):
    uri = getattr(obj, "uri", None)
    print(f"{name}: {type(obj).__name__}")
    if uri:
        print(f"  uri: {uri}")
    if hasattr(obj, "keys"):
        print(f"  keys: {list(obj.keys())}")

describe_soma_node("census", census)
describe_soma_node("census_info", census["census_info"])
describe_soma_node("census_data", census["census_data"])

In [ ]:
datasets = census['census_info']['datasets'].read()

In [ ]:
d = datasets.concat().to_pandas()

In [ ]:
d

In [ ]:
summary = census["census_info"]["summary"].read().concat().to_pandas()
summary

In [ ]:
datasets = census['census_info']['datasets'].read()

## 3. SOMA experiment layout

For an organism, SOMA gives us an `Experiment`: cell metadata (`obs`) plus one or more measurements. In this project we care about the RNA measurement.

In [ ]:
human = census["census_data"][ORGANISM]
rna = human.ms[MEASUREMENT]

describe_soma_node("human experiment", human)
describe_soma_node("human.obs", human.obs)
describe_soma_node("human.ms", human.ms)
describe_soma_node("human.ms['RNA']", rna)
describe_soma_node("human.ms['RNA'].var", rna.var)
describe_soma_node("human.ms['RNA'].X", rna.X)

## 4. Axis schemas: `obs` and `var`

These are metadata tables. Reading `var` for four genes is cheap. Enumerating `obs` for a large tissue can still take a while, so that part is guarded.

In [ ]:
print("obs schema")
print(human.obs.schema)
print("\nvar schema")
print(rna.var.schema)

In [ ]:
var_filter = "feature_name in " + repr(GENES)
adora_var = (
    rna.var.read(
        value_filter=var_filter,
        column_names=["soma_joinid", "feature_id", "feature_name", "feature_length"],
    )
    .concat()
    .to_pandas()
    .sort_values("feature_name")
)
adora_var

In [ ]:
if RUN_OBS_ENUMERATION:
    obs_probe_filter = "tissue_general == 'brain' and disease == 'normal' and is_primary_data == True"
    obs_probe = (
        human.obs.read(
            value_filter=obs_probe_filter,
            column_names=["soma_joinid", "cell_type", "tissue", "tissue_general", "dataset_id"],
        )
        .concat()
        .to_pandas()
    )
    print(f"brain normal primary obs rows: {len(obs_probe):,}")
    display(obs_probe.head())
else:
    obs_probe = pd.DataFrame()
    print("Skipped. Set RUN_OBS_ENUMERATION = True in the first cell to enumerate brain obs metadata.")

The guarded `obs_probe` read enumerates metadata only. It can still take time for a large tissue, but it is not the same cost class as reading expression matrix fragments.

In [ ]:
if not obs_probe.empty:
    cell_type_counts = obs_probe["cell_type"].value_counts().rename_axis("cell_type").reset_index(name="n_cells")
    display(cell_type_counts.head(20))
else:
    print("No obs metadata loaded yet.")

## 5. The X layer as a TileDB sparse array

`X['raw']` is the sparse cell x gene matrix. At the SOMA level, it is triples like `(cell soma_joinid, gene soma_joinid, value)`. At the TileDB level, it is an array directory with schema, commits, metadata, and fragments.

In [ ]:
x_raw = rna.X[X_LAYER]
describe_soma_node(f"X[{X_LAYER!r}]", x_raw)
print("\nSOMA schema")
print(x_raw.schema)

In [ ]:
def vfs_ls(uri, n=20):
    vfs = tiledb.VFS()
    entries = list(vfs.ls(uri))
    print(f"{uri}")
    print(f"{len(entries):,} entries")
    for entry in entries[:n]:
        print(" ", entry)
    if len(entries) > n:
        print(f"  ... {len(entries) - n:,} more")
    return entries

x_entries = vfs_ls(x_raw.uri)

In [ ]:
fragment_uri = x_raw.uri.rstrip("/") + "/__fragments"
commit_uri = x_raw.uri.rstrip("/") + "/__commits"

fragments = vfs_ls(fragment_uri, n=10)
commits = vfs_ls(commit_uri, n=10)

## 6. TileDB schema details

This is the closest view of the physical array: dimensions, attributes, tile extents, capacity, and filters. The key question from the wiki page is whether reads are naturally organized around the cell axis, the gene axis, or both.

In [ ]:
with tiledb.open(x_raw.uri, mode="r") as arr:
    schema = arr.schema
    print(schema)
    print("\nDomain dimensions")
    for dim in schema.domain:
        print(f"- {dim.name}: domain={dim.domain}, tile={dim.tile}, dtype={dim.dtype}")
    print("\nAttributes")
    for attr in schema:
        print(f"- {attr.name}: dtype={attr.dtype}, filters={attr.filters}")

## 7. Fragment directory peek

A fragment is the useful mental unit for S3 traffic. The exact file names can vary by TileDB version, but this gives a concrete sense of `__fragments/<fragment>/...` rather than treating it as abstract storage.

In [ ]:
if fragments:
    first_fragment = fragments[0]
    first_fragment_entries = vfs_ls(first_fragment, n=30)
else:
    print("No fragments listed.")

## 8. Why the four-gene query can still be expensive

The query below is intentionally disabled by default. It asks for four genes in one relatively narrow cell-type/tissue slice. The important lesson is that the returned `AnnData` can be tiny while the storage engine still had to inspect much larger cell-major fragments.

Flip `RUN_LIVE_X_QUERY` at the top only when you want to feel the timing.

In [ ]:
if RUN_LIVE_X_QUERY:
    obs_value_filter = (
        "tissue_general == 'brain' "
        "and cell_type == 'astrocyte' "
        "and disease == 'normal' "
        "and is_primary_data == True"
    )
    var_value_filter = "feature_name in " + repr(GENES)

    t0 = time.monotonic()
    adata = cellxgene_census.get_anndata(
        census=census,
        organism="Homo sapiens",
        measurement_name=MEASUREMENT,
        X_name=X_LAYER,
        obs_value_filter=obs_value_filter,
        var_value_filter=var_value_filter,
        obs_column_names=["cell_type", "tissue", "tissue_general", "dataset_id"],
    )
    dt = time.monotonic() - t0
    print(f"returned shape: {adata.shape}, nnz={adata.X.nnz:,}, seconds={dt:,.1f}")
    display(adata)
else:
    print("Skipped. Set RUN_LIVE_X_QUERY = True in the first cell to run this intentionally.")

## 9. Native iterator sketch

This is the direction hinted in the wiki page and fetch post-mortem: use axis queries and table iteration when we want checkpointable chunks rather than one monolithic `get_anndata()` call.

This cell is a sketch, not something to run casually.

In [ ]:
print(textwrap.dedent('''
with human.axis_query(
    measurement_name="RNA",
    obs_query=tiledbsoma.AxisQuery(value_filter="tissue_general == 'brain' and cell_type == 'astrocyte'"),
    var_query=tiledbsoma.AxisQuery(value_filter="feature_name in ['ADORA1', 'ADORA2A', 'ADORA2B', 'ADORA3']"),
) as query:
    for table in query.X("raw").tables():
        # table is a pyarrow Table of sparse X triples for one streamed chunk.
        # Write/checkpoint here instead of waiting for a giant AnnData.
        ...
'''))

## 10. Close handles

Run this when finished poking around.

In [ ]:
census.close()
print("closed")